# Notebook 03 — Building the Plot-Similarity Network

**Job of this notebook:** turn the plot overviews of the network corpus into a network of films, find its narrative communities and measure each film's structural position. Everything later in the thesis reads the files written here.

**Method:** all-MiniLM-L6-v2 sentence embeddings of each `overview` (character names masked) → cosine k-nearest neighbours (k = 20) → keep only mutual neighbours → largest connected component → **consensus Louvain communities** → degree, weighted degree, eigenvector, PageRank, **exact betweenness**, clustering. Setting `community_method = "single_louvain"` and `betweenness = "sampled"` gives the simpler single-run, sampled alternative for comparison.

### Design decisions
| # | Decision | Why it matters |
|---|----------|----------------|
| 1 | Reads **`df_network_corpus.csv`** from Notebook 01 (19,919 films) and checks its fingerprint against `corpus_manifest.json`. | One place (Notebook 01) decides which films are in the thesis. |
| 2 | **Embedding cache.** Vectors are matched by film id; only films without a cached vector are embedded. | A rerun takes minutes instead of re-embedding ~20,000 plots. |
| 3 | **Mutual-neighbour filter as a join** (one merge instead of a scan of the 390,000-row edge table per candidate edge). The result is identical (verified on test data). | The slowest step becomes near-instant. |
| 4 | **Labels basis + carry-over.** Before any file is overwritten, the network the manual labels were made on is saved once in `processed/network/labels_basis/`. New communities are matched to old ones by shared films, and labels are carried across where they clearly correspond. | Manual labels survive a rebuild; only changed communities need relabelling. |
| 5 | **Consensus communities.** Single Louvain runs agree only moderately (ARI ≈ 0.40 between seeds) although modularity is high (0.60): the network has strong structure, but one run draws its boundaries arbitrarily. Communities therefore come from consensus clustering over 100 runs (Lancichinetti & Fortunato, 2012), and each film gets a **membership confidence** (core vs boundary film). | Communities and their labels do not depend on one random seed. |
| 6 | **Exact betweenness** (igraph). Two independent samples of sampled betweenness share only 9 of their top-50 films, so a sampled list of bridge films is largely noise. | Bridge-film rankings are exact and reproducible. |
| 7 | **Diagnostics:** consensus reproducibility, near-duplicate edges, edges created by shared franchise/character names (e.g. *Scooby-Doo*), modularity. | Evidence that the structure reflects plots, not names or randomness. |
| 8 | One `CFG`, a safe Drive mount, graceful handling of optional files. | Reliability. |
| 9 | **Character names are masked before embedding** (spaCy person names → "someone"). Without masking, 15% of edges join films whose overviews share a character name (70× chance) and 24 communities are held together by one first name. `mask_person_names = False` gives the unmasked network. | The network compares plots, not character names. |

In [1]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 1 — CONFIG
# ═══════════════════════════════════════════════════════════════════════════
CFG = {
    "project_dir": "/content/drive/MyDrive/thesis_movie_project",

    # ── method ──────────────────────────────────────────────────────────────
    "text_column": "overview",
    # Character names are replaced before embedding, so films are linked by plot, not by a shared first name.
    # A first run of Notebook 04 found 15% of edges joining films whose overviews share a character name
    # (70× chance), and 24 communities held together by one name ("Paul", "Henry", "Alex" …).
    "mask_person_names": True,            # False reproduces the unmasked network
    "mask_replacement": "someone",        # "Paul Atreides travels to Arrakis" → "someone travels to Arrakis"
    "ner_model": "en_core_web_sm",        # spaCy model (ships with Colab; downloaded if missing)
    "sbert_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embed_batch_size": 128,
    "k_neighbors": 20,
    "min_edge_similarity": None,          # mutual k-NN already controls density
    "keep_largest_component": True,
    "louvain_resolution": 1.0,
    "random_state": 42,

    # ── communities ──────────────────────────────────────────────────────────
    "community_method": "consensus",      # "consensus" (stable) or "single_louvain" (one networkx run, seed 42)
    "consensus_runs": 100,                # Louvain runs per consensus round
    "consensus_threshold": 0.5,           # keep an edge if its films are grouped together in ≥ this share of runs
    "consensus_max_iter": 10,
    "consensus_check_seed": 7,            # repeat the whole consensus with another master seed; None to skip
    "min_community_size": 5,              # smaller fragments join their best-connected neighbouring community

    # ── betweenness ──────────────────────────────────────────────────────────
    "betweenness": "exact",               # "exact" (igraph, all shortest paths) or "sampled" (networkx, k sources)
    "betweenness_sample_size": 1000,      # used by "sampled" and by the comparison below
    "compare_with_sampled_betweenness": True,   # show how far a sampled ranking is from the exact one

    # ── diagnostics ──────────────────────────────────────────────────────────
    "near_duplicate_similarity": 0.95,    # edges at or above this cosine are near-identical texts
    "title_word_stoplist": ["The", "This", "That", "With", "From", "Into", "Over", "Under", "Part", "Movie",
                            "Film", "Story", "Return", "Rise", "Love", "Last", "First", "Night", "Life",
                            "World", "Man", "Woman", "Girl", "Boy", "King", "Queen", "Dead", "House", "Home"],

    # ── label carry-over ─────────────────────────────────────────────────────
    "carryover_keep_jaccard": 0.50,       # ≥ this: label carried automatically
    "carryover_review_jaccard": 0.30,     # between review and keep: carried but flagged for review

    "export_graphml": True,

    # ── rebuild ──────────────────────────────────────────────────────────────
    # False: if the network for this corpus already exists, the notebook stops and keeps it.
    # True: build a new network. Its communities differ slightly, so the manual labels must be redone in Notebook 04.
    "rebuild_network": False,
}
print("CONFIG loaded.")


CONFIG loaded.


In [2]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 2 — Imports, Drive mount (idempotent), paths, helpers
# ═══════════════════════════════════════════════════════════════════════════
import json, shutil, hashlib, time, warnings, random, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import networkx as nx
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
try:
    import igraph as ig
except ImportError:                       # not preinstalled on every Colab image
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "igraph"], check=True)
    import igraph as ig

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 110)
try:
    display
except NameError:
    display = print

import os, shutil, time
def mount_drive():
    """Mount Google Drive in Colab. Safe to re-run."""
    try:
        from google.colab import drive
    except ImportError:
        print("Not running in Colab — using local paths.")
        return
    if os.path.ismount("/content/drive"):          # a real Drive mount, not just a folder with that name
        print("Google Drive already mounted.")
        return
    # A plain folder at /content/drive (created while Drive was not connected) hides the Drive mount and blocks
    # mounting ("Mountpoint must not already contain files"). Move it aside — nothing on Drive is touched.
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        aside = f"/content/drive_local_leftover_{time.strftime('%Y%m%d_%H%M%S')}"
        shutil.move("/content/drive", aside)
        print(f"Moved a local leftover folder (not the Drive itself) to {aside}.")
    drive.mount("/content/drive")
mount_drive()

PROJECT_DIR  = Path(CFG["project_dir"])
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project folder not found: {PROJECT_DIR}. Check that Google Drive is mounted "
                            "(left sidebar → Files → drive/MyDrive) and that CFG['project_dir'] is right.")
CORE_DIR     = PROJECT_DIR / "processed" / "core"
NETWORK_DIR  = PROJECT_DIR / "processed" / "network"
FEATURES_DIR = PROJECT_DIR / "processed" / "features"
TABLES_DIR   = PROJECT_DIR / "outputs" / "tables"
DIAG_DIR     = TABLES_DIR / "network_diagnostics"
BASIS_DIR    = NETWORK_DIR / "labels_basis"
for d in [NETWORK_DIR, FEATURES_DIR, TABLES_DIR, DIAG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORPUS_FILE      = CORE_DIR / "df_network_corpus.csv"
MANIFEST_FILE    = CORE_DIR / "corpus_manifest.json"
EMB_TAG          = "_names_masked" if CFG["mask_person_names"] else ""   # separate caches for the two text versions
EMB_FILE         = FEATURES_DIR / f"sbert_minilm_embeddings{EMB_TAG}.npy"
EMB_CACHE_INDEX  = FEATURES_DIR / f"sbert_minilm_embedding_index{EMB_TAG}.csv"
EMB_INDEX_FILE   = FEATURES_DIR / "sbert_minilm_embedding_index.csv"     # film list Notebook 01 checks against
EMB_META_FILE    = FEATURES_DIR / f"sbert_minilm_embedding_meta{EMB_TAG}.json"
MASK_FILE        = FEATURES_DIR / "overview_person_names_masked.csv"
NODES_OUT        = NETWORK_DIR / "semantic_network_nodes_lcc.csv"
EDGES_OUT        = NETWORK_DIR / "semantic_network_edges_mutual_knn.csv"
SUMMARY_OUT      = NETWORK_DIR / "community_summary.csv"
GRAPHML_OUT      = NETWORK_DIR / "semantic_network_lcc.graphml"
AUDIT_OUT        = TABLES_DIR / "network_audit.csv"
LABELS_FILE      = TABLES_DIR / "community_manual_labeling_template_labeled.csv"

def fingerprint(ids):
    return hashlib.sha256(",".join(pd.Series(ids).astype(str)).encode()).hexdigest()[:16]

def save_diag(df, name):
    df.to_csv(DIAG_DIR / name, index=False)
    print(f"  ✓ network_diagnostics/{name}")

print("PROJECT_DIR:", PROJECT_DIR)


Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/thesis_movie_project


In [3]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3 — Load the network corpus and check it is the one Notebook 01 wrote
# ═══════════════════════════════════════════════════════════════════════════
if not CORPUS_FILE.exists():
    raise FileNotFoundError(f"{CORPUS_FILE.name} not found — run Notebook 01 first.")
df = pd.read_csv(CORPUS_FILE, low_memory=False)

REQUIRED = ["tmdb_id", "imdb_id", "title", "release_year", "primary_genre", "genres_clean", "overview",
            "overview_word_count", "runtime", "popularity", "vote_average", "vote_count",
            "revenue_clean", "log_revenue", "cast_clean", "director_name"]
missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in {CORPUS_FILE.name}: {missing}")

df[CFG["text_column"]] = df[CFG["text_column"]].fillna("").astype(str).str.strip()
if (df[CFG["text_column"]] == "").any():
    raise ValueError("Empty overviews found — the network corpus should not contain any.")

corpus_fp = fingerprint(df["tmdb_id"])
if MANIFEST_FILE.exists():
    expected_fp = json.loads(MANIFEST_FILE.read_text())["fingerprints"].get("df_network_corpus")
    status = "matches" if expected_fp == corpus_fp else f"DOES NOT MATCH ({expected_fp})"
    print(f"Corpus fingerprint {corpus_fp} {status} corpus_manifest.json")
    if expected_fp != corpus_fp:
        raise ValueError("df_network_corpus.csv differs from what Notebook 01 recorded. Re-run Notebook 01.")
else:
    print("corpus_manifest.json not found — fingerprint not checked.")

# Keep an existing network. A rebuild on another machine can change a few nearest-neighbour links and move
# films between communities, and the manual labels (Notebook 04) belong to the saved communities.
NET_MANIFEST = NETWORK_DIR / "network_manifest.json"
if (not CFG["rebuild_network"] and NODES_OUT.exists() and NET_MANIFEST.exists()
        and json.loads(NET_MANIFEST.read_text()).get("corpus_fingerprint") == corpus_fp):
    class NetworkAlreadyBuilt(Exception):
        def _render_traceback_(self):   # Jupyter/Colab show this message instead of a traceback
            return [str(self)]
    raise NetworkAlreadyBuilt(
        "✓ The network for this corpus is already built (network_manifest.json matches), so it is kept as it is.\n"
        "  Continue with Notebook 04. To build a new network, set CFG['rebuild_network'] = True "
        "(the communities then need new labels).")

df = df.reset_index(drop=True)
df["node_id"] = np.arange(len(df), dtype=int)
print(f"Network corpus: {len(df):,} films")


Corpus fingerprint 40680c1bd55fd295 matches corpus_manifest.json
Network corpus: 19,919 films


## Labels basis

The manual community labels (`community_manual_labeling_template_labeled.csv`) refer to community numbers of the network they were written for. Those numbers change whenever the network is rebuilt. The cell below saves that network's node table and the labels file **once**, in `processed/network/labels_basis/`, before anything is overwritten. The carry-over step at the end of this notebook always compares against this saved basis.

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — Save the network the manual labels belong to (only once)
# ═══════════════════════════════════════════════════════════════════════════
basis_info_file = BASIS_DIR / "basis_info.json"
if BASIS_DIR.exists() and basis_info_file.exists():
    info = json.loads(basis_info_file.read_text())
    print(f"Labels basis already saved on {info['saved_at_utc']}: {info['nodes']:,} films, "
          f"{info['labeled_communities']} labelled communities. Not overwritten.")
elif NODES_OUT.exists() and LABELS_FILE.exists():
    BASIS_DIR.mkdir(parents=True, exist_ok=True)
    for src in [NODES_OUT, SUMMARY_OUT, AUDIT_OUT, LABELS_FILE]:
        if src.exists():
            shutil.copy2(src, BASIS_DIR / src.name)
    old_nodes = pd.read_csv(BASIS_DIR / NODES_OUT.name, usecols=["tmdb_id", "community_id"])
    old_labels = pd.read_csv(BASIS_DIR / LABELS_FILE.name)
    info = {"saved_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "nodes": len(old_nodes), "communities": int(old_nodes["community_id"].nunique()),
            "labeled_communities": int(len(old_labels)),
            "nodes_fingerprint": fingerprint(old_nodes["tmdb_id"])}
    basis_info_file.write_text(json.dumps(info, indent=2))
    print(f"Saved labels basis: {info['nodes']:,} films, {info['labeled_communities']} labelled communities "
          f"→ {BASIS_DIR}")
else:
    print("No existing network + labels found — this is a first build; nothing to carry over.")


Labels basis already saved on 2026-10-06T20:35:16+00:00: 19,202 films, 23 labelled communities. Not overwritten.


## Removing character names before embedding

Sentence embeddings react strongly to proper names: two unrelated films whose heroes are both called *Paul* end up as neighbours. spaCy's named-entity recogniser finds person names in each overview and replaces them with a neutral word (`CFG['mask_replacement']`). Places, organisations and everything else are left as they are. The masked texts are saved for inspection; the original `overview` column is not changed.

In [5]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4b — Mask person names in the overviews (spaCy NER)
# ═══════════════════════════════════════════════════════════════════════════
EMBED_COL = CFG["text_column"]
EMBED_LABEL = CFG["text_column"]
n_masked = None
if CFG["mask_person_names"]:
    import spacy
    try:
        nlp = spacy.load(CFG["ner_model"], disable=["parser", "lemmatizer"])
    except OSError:
        from spacy.cli import download
        download(CFG["ner_model"])
        nlp = spacy.load(CFG["ner_model"], disable=["parser", "lemmatizer"])
    t0 = time.time()
    masked, n_masked, names_seen = [], [], []
    for doc in nlp.pipe(df[CFG["text_column"]].tolist(), batch_size=256):
        parts, last, n = [], 0, 0
        for ent in doc.ents:
            if ent.label_ == "PERSON":
                parts += [doc.text[last:ent.start_char], CFG["mask_replacement"]]
                last, n = ent.end_char, n + 1
                names_seen.append(ent.text)
        parts.append(doc.text[last:])
        masked.append("".join(parts))
        n_masked.append(n)
    df["overview_masked"] = masked
    n_masked = np.array(n_masked)
    EMBED_COL, EMBED_LABEL = "overview_masked", f"{CFG['text_column']} (person names masked)"
    print(f"Masked {n_masked.sum():,} person names in {(n_masked > 0).mean():.1%} of overviews "
          f"({(time.time() - t0) / 60:.1f} min).")
    print("Most frequent masked names:", ", ".join(pd.Series(names_seen).value_counts().head(15).index))
    ex = df.loc[n_masked > 0, ["title", CFG["text_column"], "overview_masked"]].head(3)
    for r in ex.itertuples():
        print(f"\n{r.title}\n  before: {getattr(r, CFG['text_column'])[:160]}\n  after:  {r.overview_masked[:160]}")
    df[["tmdb_id", "title"]].assign(person_names_masked=n_masked, overview_masked=df["overview_masked"]) \
        .to_csv(MASK_FILE, index=False)
    print(f"\n  ✓ processed/features/{MASK_FILE.name}")
else:
    print("Person names are NOT masked (CFG['mask_person_names'] = False) — the original text is embedded.")


Masked 36,604 person names in 69.7% of overviews (2.4 min).
Most frequent masked names: Jack, Alex, Paul, Sam, Max, David, Tom, Sarah, John, Laura, Frank, Ben, Michael, Anna, Charlie

Kate & Leopold
  before: When her scientist ex-boyfriend discovers a portal to travel through time -- and brings back a 19th-century nobleman named Leopold to prove it -- a skeptical Ka
  after:  When her scientist ex-boyfriend discovers a portal to travel through time -- and brings back a 19th-century nobleman named someone to prove it -- a skeptical so

Embodiment of Evil
  before: Released from the Mental Health Wing of São Paulo State Penitentiary after forty years, the sadistic undertaker Zé do Caixão is back on the streets, haunted by 
  after:  Released from the Mental Health Wing of São Paulo State Penitentiary after forty years, the sadistic someone do Caixão is back on the streets, haunted by ghostl

Halfaouine: Boy of the Terraces
  before: Twelve-year-old Noura dangles uncertainly in that diff

## Embeddings

A film's vector depends only on its own overview, so vectors from an earlier run can be reused for films whose text has not changed. The cache stores which model and text column produced it; if either differs from `CFG`, nothing is reused.

In [6]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — Embeddings (reuse cached vectors by film id, embed only what is missing)
# ═══════════════════════════════════════════════════════════════════════════
texts = df[EMBED_COL].tolist()
E, reused, computed = None, 0, 0

cache_index = EMB_CACHE_INDEX if EMB_CACHE_INDEX.exists() else EMB_INDEX_FILE   # older caches have only the shared index
cache_ok = EMB_FILE.exists() and cache_index.exists()
if cache_ok:
    meta = json.loads(EMB_META_FILE.read_text()) if EMB_META_FILE.exists() else \
           {"model": CFG["sbert_model"], "text_column": "overview", "note": "cache without metadata; "
            "assumed: overview column embedded with all-MiniLM-L6-v2"}
    cache_ok = meta["model"] == CFG["sbert_model"] and meta["text_column"] == EMBED_LABEL
    if not cache_ok:
        print("Cached embeddings were made with a different model or text column — not reused.")

if cache_ok:
    old_vecs = np.load(EMB_FILE)
    old_ids = pd.read_csv(cache_index, usecols=["tmdb_id"])["tmdb_id"]
    pos = pd.Series(np.arange(len(old_ids)), index=old_ids.values)
    pos = pos[~pos.index.duplicated()]
    have = df["tmdb_id"].isin(pos.index).values
    E = np.zeros((len(df), old_vecs.shape[1]), dtype=old_vecs.dtype)   # keep the cache's precision
    E[have] = old_vecs[pos.loc[df.loc[have, "tmdb_id"]].values]
    reused = int(have.sum())
    todo = np.where(~have)[0]
else:
    todo = np.arange(len(df))

if len(todo):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(CFG["sbert_model"])
    t0 = time.time()
    new_vecs = model.encode([texts[i] for i in todo], batch_size=CFG["embed_batch_size"],
                            show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
    if E is None:
        E = np.zeros((len(df), new_vecs.shape[1]), dtype=np.float32)
    E[todo] = new_vecs
    computed = len(todo)
    print(f"Embedded {computed:,} films in {(time.time() - t0) / 60:.1f} min")

E = normalize(E, norm="l2", axis=1)
np.save(EMB_FILE, E)
df[["node_id", "tmdb_id", "imdb_id", "title", "release_year"]].to_csv(EMB_INDEX_FILE, index=False)
df[["node_id", "tmdb_id", "imdb_id", "title", "release_year"]].to_csv(EMB_CACHE_INDEX, index=False)
EMB_META_FILE.write_text(json.dumps({"model": CFG["sbert_model"], "text_column": EMBED_LABEL,
                                     "films": len(df), "corpus_fingerprint": corpus_fp}, indent=2))
print(f"Embeddings ready: {E.shape} — reused {reused:,}, newly embedded {computed:,}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Embedded 19,919 films in 0.4 min
Embeddings ready: (19919, 384) — reused 0, newly embedded 19,919


## Mutual k-nearest-neighbour network

Each film lists its 20 most similar plots. An edge is kept only if the two films list each other ("mutual handshake"), which removes one-sided links to generic, hub-like plots. Edge weight is the cosine similarity of the two overviews.

In [7]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — k-NN search and directed candidate edges (vectorised)
# ═══════════════════════════════════════════════════════════════════════════
n, K = len(df), CFG["k_neighbors"]
if K >= n:
    raise ValueError("k_neighbors must be smaller than the number of films.")

t0 = time.time()
knn = NearestNeighbors(n_neighbors=K + 1, metric="cosine", algorithm="brute", n_jobs=-1).fit(E)
dist, ind = knn.kneighbors(E)
print(f"k-NN search: {(time.time() - t0) / 60:.2f} min")

# Take neighbour positions 1..K and drop any that is the film itself.
src  = np.repeat(np.arange(n), K)
dst  = ind[:, 1:].ravel()
sim  = 1.0 - dist[:, 1:].ravel()
rank = np.tile(np.arange(1, K + 1), n)
keep = src != dst
if CFG["min_edge_similarity"] is not None:
    keep &= sim >= CFG["min_edge_similarity"]
directed = pd.DataFrame({"source_node_id": src[keep], "target_node_id": dst[keep],
                         "cosine_similarity": sim[keep], "knn_rank": rank[keep]})
print(f"Directed candidate edges: {len(directed):,}")


k-NN search: 0.15 min
Directed candidate edges: 398,367


In [8]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 7 — Mutual filter as a join (one merge, not a scan per edge)
# ═══════════════════════════════════════════════════════════════════════════
reverse = directed.rename(columns={"source_node_id": "target_node_id", "target_node_id": "source_node_id",
                                   "cosine_similarity": "reverse_similarity", "knn_rank": "target_rank"})
edges = (directed[directed["source_node_id"] < directed["target_node_id"]]
         .merge(reverse, on=["source_node_id", "target_node_id"], how="inner")
         .rename(columns={"knn_rank": "source_rank"}))
edges["cosine_similarity"] = (edges["cosine_similarity"] + edges["reverse_similarity"]) / 2
edges = (edges.sort_values(["source_node_id", "source_rank"], kind="stable")
              [["source_node_id", "target_node_id", "cosine_similarity", "source_rank", "target_rank"]]
              .reset_index(drop=True))
if edges.empty:
    raise ValueError("No mutual edges — increase k_neighbors or remove min_edge_similarity.")
print(f"Mutual k-NN edges: {len(edges):,}")


Mutual k-NN edges: 70,372


In [9]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 8 — Graph and largest connected component
# ═══════════════════════════════════════════════════════════════════════════
G = nx.Graph()
node_attr_cols = ["tmdb_id", "imdb_id", "title", "release_year", "primary_genre", "vote_count",
                  "vote_average", "popularity"]
attrs = df[["node_id"] + node_attr_cols].copy()
for c in ["tmdb_id", "imdb_id", "title", "primary_genre"]:
    attrs[c] = attrs[c].astype(str)
G.add_nodes_from((int(r["node_id"]), {c: r[c] for c in node_attr_cols}) for r in attrs.to_dict("records"))
G.add_edges_from(
    (int(s), int(t), {"weight": float(w), "cosine_similarity": float(w), "source_rank": int(sr), "target_rank": int(tr)})
    for s, t, w, sr, tr in edges.itertuples(index=False, name=None))

components = sorted(nx.connected_components(G), key=len, reverse=True)
component_sizes = [len(c) for c in components]
G_lcc = G.subgraph(components[0]).copy() if CFG["keep_largest_component"] else G.copy()

print(f"Full graph: {G.number_of_nodes():,} films, {G.number_of_edges():,} edges, "
      f"{len(component_sizes):,} components")
print(f"Largest component: {G_lcc.number_of_nodes():,} films "
      f"({G_lcc.number_of_nodes() / G.number_of_nodes():.2%}), {G_lcc.number_of_edges():,} edges, "
      f"average degree {2 * G_lcc.number_of_edges() / G_lcc.number_of_nodes():.2f}")


Full graph: 19,919 films, 70,372 edges, 772 components
Largest component: 19,084 films (95.81%), 70,252 edges, average degree 7.36


## Communities: consensus over many Louvain runs

Louvain is randomised, and on this network single runs disagree considerably (ARI ≈ 0.40 between seeds) even though every run finds strong structure (modularity ≈ 0.60). Many different partitions are almost equally good, so one run draws the boundaries of the broad drama/comedy neighbourhoods arbitrarily.

**Consensus clustering** (Lancichinetti & Fortunato, 2012) removes that arbitrariness:
1. Run Louvain 100 times.
2. For every edge, record the share of runs in which its two films land in the same community.
3. Keep edges with a share ≥ 0.5, weight them by that share, and repeat until every run agrees.
4. Fragments smaller than 5 films join the neighbouring community they are most strongly linked to. Communities are numbered by size (0 = largest).

Two outputs follow:
* **Membership confidence** (per film): on average across the first 100 runs, the share of the film's consensus community that sits in the same run-community as the film. High = core film, low = boundary film whose plot sits between neighbourhoods.
* **Community robustness** (per community): the mean membership confidence of its films.

The whole procedure is repeated with a second master seed; the two consensus partitions should be nearly identical.

In [10]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 9 — Communities (consensus Louvain, or a single run), modularity
# ═══════════════════════════════════════════════════════════════════════════
order  = list(G_lcc.nodes())                         # one fixed node order for all arrays below
pos_of = {v: i for i, v in enumerate(order)}
N      = len(order)
E_idx  = np.array([(pos_of[u], pos_of[v]) for u, v in G_lcc.edges()], dtype=np.int64)
W      = np.array([d["weight"] for _, _, d in G_lcc.edges(data=True)])

def louvain_runs(edge_idx, weights, n_runs, rng):
    g = ig.Graph(n=N, edges=edge_idx.tolist())
    L = np.empty((n_runs, N), dtype=np.int64)
    for r in range(n_runs):
        random.seed(int(rng.integers(2**31 - 1)))   # igraph draws from Python's random module
        L[r] = g.community_multilevel(weights=weights.tolist(), resolution=CFG["louvain_resolution"]).membership
    return L

def consensus_partition(master_seed):
    rng = np.random.default_rng(master_seed)
    edges_c, w_c, first, unresolved = E_idx, W, None, 1.0
    for it in range(1, CFG["consensus_max_iter"] + 1):
        L = louvain_runs(edges_c, w_c, CFG["consensus_runs"], rng)
        first = L if first is None else first
        agree = (L[:, edges_c[:, 0]] == L[:, edges_c[:, 1]]).mean(axis=0)
        unresolved = float(((agree > 0) & (agree < 1)).mean())
        if unresolved == 0:
            break
        keep = agree >= CFG["consensus_threshold"]
        edges_c, w_c = edges_c[keep], agree[keep]
    return L[0].copy(), first, it, unresolved

def merge_small(labels):
    labels, moved = labels.copy(), 0
    for _ in range(25):
        sizes = np.bincount(labels)
        small = sizes[labels] < CFG["min_community_size"]
        if not small.any():
            break
        a, b = E_idx[:, 0], E_idx[:, 1]
        e = pd.DataFrame({"node": np.r_[a, b], "target": np.r_[labels[b], labels[a]], "w": np.r_[W, W]})
        e = e[small[e["node"].values] & ~small[np.r_[b, a]]]
        if e.empty:
            break
        best = (e.groupby(["node", "target"])["w"].sum().reset_index()
                 .sort_values("w", ascending=False).drop_duplicates("node"))
        labels[best["node"].values] = best["target"].values
        moved += len(best)
    return labels, moved

def renumber_by_size(labels):
    sizes = pd.Series(labels).value_counts(sort=True)
    return pd.Series(labels).map({old: new for new, old in enumerate(sizes.index)}).to_numpy()

def membership_confidence(final, runs):
    sizes, conf = np.bincount(final), np.zeros(N)
    for r in range(runs.shape[0]):
        key = final * (runs[r].max() + 1) + runs[r]
        _, inv, cnt = np.unique(key, return_inverse=True, return_counts=True)
        conf += cnt[inv] / sizes[final]
    return conf / runs.shape[0]

t0 = time.time()
consensus_diag, confidence = None, np.full(N, np.nan)
if CFG["community_method"] == "consensus":
    raw, first_runs, n_iter, unresolved = consensus_partition(CFG["random_state"])
    merged, moved = merge_small(raw)
    final = renumber_by_size(merged)
    confidence = membership_confidence(final, first_runs)
    single_ari = np.array([adjusted_rand_score(final, first_runs[r]) for r in range(len(first_runs))])
    rows = [("consensus rounds until all runs agreed", n_iter),
            ("share of edges still undecided at the end", round(unresolved, 4)),
            ("films moved from fragments < min_community_size", moved),
            ("communities", int(final.max() + 1)),
            ("single runs vs consensus: mean ARI", round(single_ari.mean(), 3)),
            ("single runs vs consensus: min–max ARI", f"{single_ari.min():.3f}–{single_ari.max():.3f}")]
    if CFG["consensus_check_seed"] is not None:
        check = renumber_by_size(merge_small(consensus_partition(CFG["consensus_check_seed"])[0])[0])
        rows += [("reproducibility: ARI vs consensus with another master seed", round(adjusted_rand_score(final, check), 4)),
                 ("reproducibility: NMI vs consensus with another master seed", round(normalized_mutual_info_score(final, check), 4))]
    consensus_diag = pd.DataFrame(rows, columns=["measure", "value"])
    partition = {order[i]: int(final[i]) for i in range(N)}
else:   # one networkx Louvain run
    comms = nx.community.louvain_communities(G_lcc, weight="weight", seed=CFG["random_state"],
                                             resolution=CFG["louvain_resolution"])
    partition = {node: cid for cid, members in enumerate(comms) for node in members}

nx.set_node_attributes(G_lcc, partition, "community_id")
labels_arr = np.array([partition[v] for v in order])
communities = [set(np.array(order)[labels_arr == c]) for c in sorted(set(labels_arr))]
modularity = nx.community.modularity(G_lcc, communities, weight="weight")
confidence_s = pd.Series(confidence, index=order)
robustness = confidence_s.groupby(labels_arr).mean().rename("community_robustness")
print(f"{CFG['community_method']}: {len(communities)} communities, modularity {modularity:.3f} "
      f"({(time.time() - t0) / 60:.1f} min)")
if consensus_diag is not None:
    consensus_diag.loc[len(consensus_diag)] = ["modularity of consensus partition", round(modularity, 4)]
    display(consensus_diag)
    save_diag(consensus_diag, "consensus_communities.csv")


consensus: 105 communities, modularity 0.651 (4.0 min)


,measure,value
0,consensus rounds until all runs agreed,5
1,share of edges still undecided at the end,0.0
2,films moved from fragments < min_community_size,113
3,communities,105
4,single runs vs consensus: mean ARI,0.52
5,single runs vs consensus: min–max ARI,0.424–0.596
6,reproducibility: ARI vs consensus with another master seed,0.8736
7,reproducibility: NMI vs consensus with another master seed,0.9145
8,modularity of consensus partition,0.6509


  ✓ network_diagnostics/consensus_communities.csv


In [11]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 10 — Centrality and position metrics
# ═══════════════════════════════════════════════════════════════════════════
t0 = time.time()
degree_d          = dict(G_lcc.degree())
weighted_degree_d = dict(G_lcc.degree(weight="weight"))
degree_cent_d     = nx.degree_centrality(G_lcc)
try:
    eigen_d = nx.eigenvector_centrality(G_lcc, max_iter=500, weight="weight")
except Exception as e:
    print("Eigenvector centrality failed, using NaN:", e)
    eigen_d = {v: np.nan for v in G_lcc}
pagerank_d   = nx.pagerank(G_lcc, weight="weight", alpha=0.85)
clustering_d = nx.clustering(G_lcc, weight="weight")
k_between    = min(CFG["betweenness_sample_size"], N)

def sampled_betweenness(seed):     # networkx, k sampled source films, unweighted
    return nx.betweenness_centrality(G_lcc, k=k_between, weight=None, seed=seed)

if CFG["betweenness"] == "exact":   # all shortest paths, unweighted (standard definition), igraph for speed
    raw_b = np.array(ig.Graph(n=N, edges=E_idx.tolist()).betweenness(directed=False))
    between_d = dict(zip(order, raw_b * 2 / ((N - 1) * (N - 2))))   # same scale as networkx normalized=True
else:
    between_d = sampled_betweenness(CFG["random_state"])
print(f"Centralities: {(time.time() - t0) / 60:.1f} min  (betweenness: {CFG['betweenness']})")

between_check = None
if CFG["betweenness"] == "exact" and CFG["compare_with_sampled_betweenness"]:
    a, b = pd.Series(between_d), pd.Series(sampled_betweenness(CFG["random_state"]))
    top = lambda x, n: set(x.sort_values(ascending=False).head(n).index)
    between_check = pd.DataFrame([
        {"measure": f"Spearman ρ, exact vs sampled (k={k_between})", "value": round(a.corr(b, method="spearman"), 3)},
        {"measure": "top-50 bridge films shared", "value": f"{len(top(a, 50) & top(b, 50))}/50"},
        {"measure": "top-200 bridge films shared", "value": f"{len(top(a, 200) & top(b, 200))}/200"},
    ])
    display(between_check)
    save_diag(between_check, "betweenness_exact_vs_sampled.csv")


Centralities: 1.0 min  (betweenness: exact)


,measure,value
0,"Spearman ρ, exact vs v1 sampled (k=1000)",0.94
1,top-50 bridge films shared,17/50
2,top-200 bridge films shared,97/200


  ✓ network_diagnostics/betweenness_exact_vs_sampled.csv


In [12]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 11 — Node, edge and community tables (stable schemas, new columns at the end)
# ═══════════════════════════════════════════════════════════════════════════
BASE_COLS = ["node_id", "tmdb_id", "imdb_id", "title", "release_year", "primary_genre", "genres_clean",
             "overview_word_count", "runtime", "popularity", "vote_average", "vote_count",
             "revenue_clean", "log_revenue", "cast_clean", "director_name"]
nodes_lcc = pd.DataFrame({"node_id": list(G_lcc.nodes())}).merge(df[BASE_COLS], on="node_id", how="left")
metrics = {"community_id": partition, "degree": degree_d, "weighted_degree": weighted_degree_d,
           "degree_centrality": degree_cent_d, "eigenvector_centrality": eigen_d, "pagerank": pagerank_d,
           "betweenness_centrality_sampled": between_d, "clustering_coefficient": clustering_d}
for col, d in metrics.items():
    nodes_lcc[col] = nodes_lcc["node_id"].map(d)
for m in ["degree", "weighted_degree", "degree_centrality", "eigenvector_centrality", "pagerank",
          "betweenness_centrality_sampled"]:
    nodes_lcc[f"{m}_rank"] = nodes_lcc[m].rank(ascending=False, method="min").astype(int)
nodes_lcc["betweenness_centrality"] = nodes_lcc["node_id"].map(between_d)   # same values; clearer name
nodes_lcc["membership_confidence"] = nodes_lcc["node_id"].map(confidence_s).round(3)
nodes_lcc["community_robustness"] = nodes_lcc["community_id"].map(robustness).round(3)

lcc_set = set(nodes_lcc["node_id"])
edges_lcc = edges[edges["source_node_id"].isin(lcc_set) & edges["target_node_id"].isin(lcc_set)].copy()
title_map, year_map = df.set_index("node_id")["title"], df.set_index("node_id")["release_year"]
edges_lcc["source_title"] = edges_lcc["source_node_id"].map(title_map)
edges_lcc["target_title"] = edges_lcc["target_node_id"].map(title_map)
edges_lcc["source_year"]  = edges_lcc["source_node_id"].map(year_map)
edges_lcc["target_year"]  = edges_lcc["target_node_id"].map(year_map)
edges_lcc = edges_lcc[["source_node_id", "target_node_id", "source_title", "target_title", "source_year",
                       "target_year", "cosine_similarity", "source_rank", "target_rank"]]

reps = (nodes_lcc.sort_values("weighted_degree", ascending=False).groupby("community_id").head(8)
          .assign(label=lambda d: d["title"] + " (" + d["release_year"].astype("Int64").astype(str) + ")")
          .groupby("community_id")["label"].agg("; ".join).rename("representative_films"))
community_summary = (nodes_lcc.groupby("community_id")
    .agg(n_films=("node_id", "count"), median_year=("release_year", "median"),
         min_year=("release_year", "min"), max_year=("release_year", "max"),
         median_vote_count=("vote_count", "median"), median_vote_average=("vote_average", "median"),
         median_revenue=("revenue_clean", "median"),
         top_genre=("primary_genre", lambda x: x.value_counts().index[0] if x.notna().any() else np.nan))
    .join(reps).join(robustness.round(3)).reset_index().sort_values("n_films", ascending=False))

print(f"nodes_lcc {nodes_lcc.shape} · edges_lcc {edges_lcc.shape} · communities {len(community_summary)}")
display(community_summary.head(10))


nodes_lcc (19084, 33) · edges_lcc (70252, 9) · communities 105


,community_id,n_films,median_year,min_year,max_year,median_vote_count,median_vote_average,median_revenue,top_genre,representative_films,community_robustness
0,0,1700,2011.0,1990,2024,107.0,6.1000,9945651.5,Action,The Postcard Killings (2020); Varg Veum - Fallen Angels (2008); Kite (2014); Beyond the Law (2019); Alex C...,0.669
1,1,1360,2012.0,1990,2024,87.0,6.0000,6069605.0,Comedy,Obsession (2019); The Salton Sea (2002); Amanda (2018); Remember Me (2010); Abraham's Valley (1993); Bihte...,0.597
2,2,1346,2014.0,1990,2024,88.5,6.1000,5101936.5,Drama,Beautiful Kate (2009); Glass House: The Good Mother (2006); The Kindred (2021); First Case (2024); The Hou...,0.517
3,3,1246,2013.0,1990,2024,95.5,6.3295,6995196.5,Drama,Manhunt (2012); Europa Europa (1990); Sobibor (2018); Ernest in the Army (1998); War of the Dead (2011); D...,0.657
4,4,1120,2013.0,1990,2024,155.5,6.1000,36850101.0,Action,Assassin 33 A.D. (2020); The Visitor from the Future (2022); The Tomorrow War (2021); Justice League: Warw...,0.780
5,5,1079,2013.0,1990,2024,95.0,5.7000,7523353.0,Horror,The Invoking (2013); Eko Eko Azarak: Wizard of Darkness (1995); Asmodexia (2014); The Apparition (2012); H...,0.519
6,6,816,2012.0,1990,2024,145.0,6.4000,25314960.0,Drama,Godmothered (2020); Legend of the Millennium Dragon (2011); Hagen (2024); The Swan Princess: A Fairytale I...,0.669
7,7,717,2014.0,1990,2024,91.0,6.1000,3770657.0,Drama,"Love Affair(s) (2020); The Past (2013); Le fils préféré (1994); Au Revoir Taipei (2010); Mektoub, My Love:...",0.599
8,8,699,2013.0,1990,2024,71.0,6.7000,10315875.0,Drama,Shanghai Kiss (2007); Samurai Cop (1991); Ley Lines (1999); Shoplifters (2018); Sakra (2023); Into the Sun...,0.740
9,9,656,2012.0,1990,2024,110.5,6.3000,10912086.5,Comedy,Ma che bella sorpresa (2015); Thesis on a Homicide (2013); The Substitute (2022); After (2019); 18 Years O...,0.491


## Edge diagnostics — is the similarity about plots?

* **Near-duplicate edges:** pairs whose overviews are almost the same text (cosine ≥ 0.95). These are usually re-releases, duplicate entries or copied descriptions, not narrative similarity.
* **Shared-name edges:** pairs whose titles share a proper name (*Scooby-Doo*, *Batman*, *Rocky*) — franchise or character names that also appear in the overviews. Some of these links are genuine story continuity, but they are driven by names rather than plot structure. Their share shows how much of the network rests on names.
* **Within-community share:** the fraction of edges that stay inside a community — a simple measure of how clearly the communities are separated.

In [14]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 12 — Near-duplicate edges, shared-name edges, community separation
# ═══════════════════════════════════════════════════════════════════════════
E_ = edges_lcc.copy()
near_dup = E_[E_["cosine_similarity"] >= CFG["near_duplicate_similarity"]] \
             .sort_values("cosine_similarity", ascending=False)

names = (nodes_lcc.set_index("node_id")["title"].fillna("")
           .str.findall(r"\b[A-Z][A-Za-z'’\-]{3,}\b").explode().dropna())
names = names[~names.isin(CFG["title_word_stoplist"])].rename("name").reset_index()
shared = (E_[["source_node_id", "target_node_id"]].reset_index()
            .merge(names.rename(columns={"node_id": "source_node_id"}), on="source_node_id")
            .merge(names.rename(columns={"node_id": "target_node_id"}), on=["target_node_id", "name"]))
shared_edge_idx = shared["index"].unique()
name_counts = shared.drop_duplicates(["index", "name"])["name"].value_counts().rename_axis("name") \
                    .reset_index(name="edges")

comm = nodes_lcc.set_index("node_id")["community_id"]
within = (E_["source_node_id"].map(comm) == E_["target_node_id"].map(comm))

edge_diag = pd.DataFrame([
    {"measure": "edges in largest component", "value": len(E_)},
    {"measure": f"near-duplicate edges (cosine ≥ {CFG['near_duplicate_similarity']})", "value": len(near_dup)},
    {"measure": "edges between films whose titles share a proper name", "value": len(shared_edge_idx)},
    {"measure": "share of edges driven by a shared title name (%)", "value": round(len(shared_edge_idx) / len(E_) * 100, 2)},
    {"measure": "share of edges inside a community (%)", "value": round(within.mean() * 100, 2)},
    {"measure": "share of shared-name edges inside a community (%)",
     "value": round(within.loc[shared_edge_idx].mean() * 100, 2) if len(shared_edge_idx) else np.nan},
])
display(edge_diag)
print("\nNames creating the most edges:"); display(name_counts.head(15))
print("\nClosest near-duplicate pairs:"); display(near_dup.head(10))
save_diag(edge_diag, "edge_diagnostics_summary.csv")
save_diag(near_dup, "near_duplicate_edges.csv")
save_diag(name_counts, "shared_title_name_edges.csv")


,measure,value
0,edges in largest component,70252.00
1,near-duplicate edges (cosine ≥ 0.95),3.00
2,edges between films whose titles share a proper name,2481.00
3,share of edges driven by a shared title name (%),3.53
4,share of edges inside a community (%),69.27
5,share of shared-name edges inside a community (%),88.07



Names creating the most edges:


,name,edges
0,Scooby-Doo,176
1,Christmas,112
2,Shark,71
3,Batman,69
4,Conan,52
5,Detective,48
6,League,45
7,Justice,42
8,Dragon,37
9,Naruto,35



Closest near-duplicate pairs:


,source_node_id,target_node_id,source_title,target_title,source_year,target_year,cosine_similarity,source_rank,target_rank
8090,1227,2969,A Christmas Carol,A Christmas Carol,1997,1999,1.000000,1,2
61769,13164,19879,"Batman: The Dark Knight Returns, Part 1",Batman: The Dark Knight Returns,2012,2013,0.995399,1,1
4456,676,2218,Nineties Part II,Nineties,1993,1992,0.994571,1,1


  ✓ network_diagnostics/edge_diagnostics_summary.csv
  ✓ network_diagnostics/near_duplicate_edges.csv
  ✓ network_diagnostics/shared_title_name_edges.csv


## Carrying the manual labels across

New communities are matched to the communities in the labels basis by the films they share (Jaccard overlap = shared films ÷ films in either).
* **carried** — overlap ≥ 0.50: same community, label copied.
* **carried – review** — overlap 0.30–0.50: label copied as a starting point; check it.
* **new** — overlap < 0.30: no clear predecessor; needs a fresh label.
* A note marks old communities that **split** into several new ones.

The labels file is never overwritten. The result is a draft (`community_labels_draft_v2.csv`) for Notebook 04.

In [15]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 13 — Label carry-over from the labels basis
# ═══════════════════════════════════════════════════════════════════════════
carryover, draft = None, None
basis_nodes_file, basis_labels_file = BASIS_DIR / NODES_OUT.name, BASIS_DIR / LABELS_FILE.name
if basis_nodes_file.exists() and basis_labels_file.exists():
    old = pd.read_csv(basis_nodes_file, usecols=["tmdb_id", "community_id"]).rename(columns={"community_id": "old_cid"})
    old_labels = pd.read_csv(basis_labels_file)
    label_col = next(c for c in ["suggested_label_manual", "narrative_label", "label"] if c in old_labels.columns)
    new = nodes_lcc[["tmdb_id", "community_id"]].rename(columns={"community_id": "new_cid"})

    overlap = new.merge(old, on="tmdb_id").groupby(["new_cid", "old_cid"]).size().rename("shared").reset_index()
    overlap = overlap.merge(new["new_cid"].value_counts().rename("new_size"), left_on="new_cid", right_index=True) \
                     .merge(old["old_cid"].value_counts().rename("old_size"), left_on="old_cid", right_index=True)
    overlap["jaccard"] = overlap["shared"] / (overlap["new_size"] + overlap["old_size"] - overlap["shared"])
    overlap["share_of_old"] = overlap["shared"] / overlap["old_size"]
    overlap = overlap.sort_values(["new_cid", "jaccard"], ascending=[True, False])
    best = overlap.groupby("new_cid").head(1).set_index("new_cid")
    second = overlap.groupby("new_cid").nth(1).set_index("new_cid")

    carryover = pd.DataFrame({"community_id": new["new_cid"].value_counts().sort_index().index})
    carryover["n_films"] = carryover["community_id"].map(new["new_cid"].value_counts())
    carryover["matched_old_community"] = carryover["community_id"].map(best["old_cid"])
    carryover["jaccard"] = carryover["community_id"].map(best["jaccard"]).round(3)
    carryover["share_of_old_community"] = carryover["community_id"].map(best["share_of_old"]).round(3)
    carryover["second_old_community"] = carryover["community_id"].map(second["old_cid"])
    carryover["second_jaccard"] = carryover["community_id"].map(second["jaccard"]).round(3)
    carryover["status"] = np.select(
        [carryover["jaccard"] >= CFG["carryover_keep_jaccard"], carryover["jaccard"] >= CFG["carryover_review_jaccard"]],
        ["carried", "carried – review"], default="new")
    split_old = carryover.loc[carryover["status"] != "new", "matched_old_community"].value_counts()
    split_old = split_old[split_old > 1].index
    carryover["note"] = np.where(carryover["matched_old_community"].isin(split_old) & (carryover["status"] != "new"),
                                 "old community split into several new ones", "")

    lab = old_labels.drop_duplicates("community_id").set_index("community_id")
    carryover["has_label"] = carryover["matched_old_community"].isin(lab.index) & (carryover["status"] != "new")
    carryover["carried_label"] = np.where(carryover["has_label"],
                                          carryover["matched_old_community"].map(lab[label_col]), "")
    carryover.loc[(carryover["status"] != "new") & ~carryover["has_label"], "note"] = \
        "matched old community had no manual label"
    carryover = (carryover.merge(community_summary[["community_id", "representative_films", "community_robustness"]],
                                 on="community_id", how="left")
                          .sort_values("n_films", ascending=False).reset_index(drop=True))

    # Draft labels file in the same shape as the manual one (manual file is never touched)
    draft = pd.DataFrame({"community_id": carryover["community_id"], "n_films": carryover["n_films"]})
    for c in old_labels.columns:
        if c in ("community_id", "n_films"):
            continue
        draft[c] = np.where(carryover["has_label"], carryover["matched_old_community"].map(lab[c]), "")
    draft["carryover_status"] = carryover["status"].values
    draft["matched_old_community"] = carryover["matched_old_community"].values
    draft["jaccard"] = carryover["jaccard"].values
    draft["representative_films"] = carryover["representative_films"].values

    print("Carry-over status:"); display(carryover["status"].value_counts().to_frame("communities"))
    print(f"Films in communities whose label was carried: "
          f"{carryover.loc[carryover['status'] == 'carried', 'n_films'].sum() / len(nodes_lcc):.1%}")
    display(carryover[["community_id", "n_films", "status", "matched_old_community", "jaccard",
                       "carried_label", "note"]].head(30))
    carryover.to_csv(TABLES_DIR / "community_label_carryover.csv", index=False)
    draft.to_csv(TABLES_DIR / "community_labels_draft_v2.csv", index=False)
    print("  ✓ community_label_carryover.csv  ✓ community_labels_draft_v2.csv")
else:
    print("No labels basis — skipping carry-over (first build).")


Carry-over status:


,communities
status,
new,101
carried – review,3
carried,1


Films in communities whose label was carried: 0.2%


,community_id,n_films,status,matched_old_community,jaccard,carried_label,note
0,0,1700,new,23,0.147,,
1,1,1360,new,14,0.117,,
2,2,1346,new,14,0.151,,
3,3,1246,carried – review,4,0.352,"War, historical conflict, and political/action drama",
4,4,1120,carried – review,11,0.332,"Science-fiction, infection, dystopia, and altered-reality narratives",
5,5,1079,new,8,0.271,,
6,6,816,new,19,0.252,,
7,7,717,new,24,0.171,,
8,8,699,new,12,0.197,,
9,9,656,new,16,0.151,,


  ✓ community_label_carryover.csv  ✓ community_labels_draft_v2.csv


In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 14 — Audit table and exports (stable file names)
# ═══════════════════════════════════════════════════════════════════════════
audit_rows = [
    ("input_network_corpus_films", len(df)),
    ("corpus_fingerprint", corpus_fp),
    ("sbert_model", CFG["sbert_model"]),
    ("embedded_text_column", EMBED_LABEL),
    ("person_names_masked", int(n_masked.sum()) if n_masked is not None else 0),
    ("overviews_with_masked_names_pct", round((n_masked > 0).mean() * 100, 2) if n_masked is not None else 0),
    ("embeddings_reused_from_cache", reused),
    ("embeddings_newly_computed", computed),
    ("k_neighbors", K),
    ("min_edge_similarity", str(CFG["min_edge_similarity"])),
    ("directed_knn_edges", len(directed)),
    ("mutual_knn_edges_full_graph", G.number_of_edges()),
    ("full_graph_nodes", G.number_of_nodes()),
    ("full_graph_components", len(component_sizes)),
    ("largest_component_nodes", G_lcc.number_of_nodes()),
    ("largest_component_edges", G_lcc.number_of_edges()),
    ("largest_component_share", round(G_lcc.number_of_nodes() / G.number_of_nodes(), 4)),
    ("average_degree_lcc", round(2 * G_lcc.number_of_edges() / G_lcc.number_of_nodes(), 4)),
    ("density_lcc", round(nx.density(G_lcc), 8)),
    ("community_method", "consensus_louvain_igraph" if CFG["community_method"] == "consensus" else "networkx_louvain"),
    ("random_state", CFG["random_state"]),
    ("louvain_resolution", CFG["louvain_resolution"]),
    ("consensus_runs", CFG["consensus_runs"] if CFG["community_method"] == "consensus" else ""),
    ("consensus_threshold", CFG["consensus_threshold"] if CFG["community_method"] == "consensus" else ""),
    ("n_communities_lcc", int(nodes_lcc["community_id"].nunique())),
    ("modularity", round(modularity, 4)),
    ("consensus_reproducibility_ARI",
     consensus_diag.set_index("measure")["value"].get("reproducibility: ARI vs consensus with another master seed", "")
     if consensus_diag is not None else ""),
    ("betweenness_method", "exact_igraph" if CFG["betweenness"] == "exact" else f"sampled_networkx_k{k_between}"),
    ("betweenness_column_note", "betweenness_centrality_sampled holds the same values as betweenness_centrality; "
                                "the old name is kept so Notebooks 04–05 and the bricks keep working"),
    ("share_edges_shared_title_name_pct", edge_diag.iloc[3]["value"]),
    ("near_duplicate_edges", len(near_dup)),
]
network_audit = pd.DataFrame(audit_rows, columns=["metric", "value"])
display(network_audit)

nodes_lcc.to_csv(NODES_OUT, index=False)
edges_lcc.to_csv(EDGES_OUT, index=False)
community_summary.to_csv(SUMMARY_OUT, index=False)
network_audit.to_csv(AUDIT_OUT, index=False)
print("  ✓", NODES_OUT.name, "  ✓", EDGES_OUT.name, "  ✓", SUMMARY_OUT.name, "  ✓", AUDIT_OUT.name)

if CFG["export_graphml"]:
    G_export = G_lcc.copy()
    for _, a in G_export.nodes(data=True):
        for k, v in list(a.items()):
            if v is None or (isinstance(v, float) and np.isnan(v)):
                a[k] = ""
    nx.write_graphml(G_export, GRAPHML_OUT)
    print("  ✓", GRAPHML_OUT.name)

(NETWORK_DIR / "network_manifest.json").write_text(json.dumps({
    "built_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "corpus_fingerprint": corpus_fp, "nodes_fingerprint": fingerprint(nodes_lcc["tmdb_id"]),
    "config": {k: v for k, v in CFG.items() if k != "project_dir"},
    "lcc_nodes": G_lcc.number_of_nodes(), "lcc_edges": G_lcc.number_of_edges(),
    "communities": int(nodes_lcc["community_id"].nunique()), "modularity": round(modularity, 4)}, indent=2, default=str))
print("  ✓ network_manifest.json")
print("\nCheck that CFG['allow_corpus_change'] is False in Notebook 01, so its safety guard protects this network.")


,metric,value
0,input_network_corpus_films,19919
1,corpus_fingerprint,40680c1bd55fd295
2,sbert_model,sentence-transformers/all-MiniLM-L6-v2
3,embedded_text_column,overview (person names masked)
4,person_names_masked,36604
5,overviews_with_masked_names_pct,69.7
6,embeddings_reused_from_cache,0
7,embeddings_newly_computed,19919
8,k_neighbors,20
9,min_edge_similarity,None


  ✓ semantic_network_nodes_lcc.csv   ✓ semantic_network_edges_mutual_knn.csv   ✓ community_summary.csv   ✓ network_audit.csv
  ✓ semantic_network_lcc.graphml
  ✓ network_manifest.json

Check that CFG['allow_corpus_change'] is False in Notebook 01 v2, so its safety guard protects this network.


In [17]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 15 — Sanity checks
# ═══════════════════════════════════════════════════════════════════════════
cols = ["title", "release_year", "primary_genre", "vote_count", "community_id"]
print("Highest weighted degree:")
display(nodes_lcc.sort_values("weighted_degree", ascending=False)[cols + ["weighted_degree"]].head(15))
print("Highest sampled betweenness:")
display(nodes_lcc.sort_values("betweenness_centrality_sampled", ascending=False)
        [cols + ["betweenness_centrality_sampled"]].head(15))
if nodes_lcc["membership_confidence"].notna().any():
    print("Well-known boundary films (lowest membership confidence among films with ≥ 1,000 votes):")
    display(nodes_lcc[nodes_lcc["vote_count"] >= 1000].sort_values("membership_confidence")
            [cols + ["membership_confidence"]].head(15))
print("Least robust communities (interpret with care):")
display(community_summary.sort_values("community_robustness")[["community_id", "n_films", "community_robustness",
                                                              "representative_films"]].head(8))


Highest weighted degree:


,title,release_year,primary_genre,vote_count,community_id,weighted_degree
14762,Scooby-Doo's Creepiest Capers,2000,Animation,37.0,41,14.269744
4995,The Return,2006,Horror,650.0,16,14.142814
15246,The Postcard Killings,2020,Drama,647.0,0,14.034301
16846,Obsession,2019,Thriller,27.0,1,13.939037
8043,Beautiful Kate,2009,Drama,68.0,2,13.925760
5721,Glass House: The Good Mother,2006,Drama,91.0,2,13.909829
10241,All That Glitters,2010,Comedy,756.0,27,13.840334
6616,Varg Veum - Fallen Angels,2008,Crime,38.0,0,13.821769
11774,Kite,2014,Drama,270.0,0,13.806968
6774,The Kindred,2021,Horror,125.0,2,13.802756


Highest sampled betweenness:


,title,release_year,primary_genre,vote_count,community_id,betweenness_centrality_sampled
18688,Black Panther: Wakanda Forever,2022,Action,7422.0,8,0.002203
4278,Alone in the Dark,2005,Action,654.0,0,0.001987
11494,The Last Days,2013,Science Fiction,375.0,24,0.001976
7733,Dune,2021,Science Fiction,14779.0,4,0.001931
13205,Mr. Kaplan,2014,Drama,26.0,3,0.001923
326,Fist of Fury 1991 II,1992,Action,31.0,2,0.001905
17388,The Ciambra,2017,Crime,172.0,20,0.001871
10449,Russendisko,2012,Comedy,57.0,3,0.001829
18182,Shoplifters,2018,Drama,2273.0,8,0.001787
14549,Casa Grande,2014,Drama,123.0,10,0.001754


Well-known boundary films (lowest membership confidence among films with ≥ 1,000 votes):


,title,release_year,primary_genre,vote_count,community_id,membership_confidence
18554,Freaks,2019,Science Fiction,1257.0,1,0.017
5663,Suburbicon,2017,Thriller,1799.0,2,0.029
11602,Pain & Gain,2013,Comedy,4783.0,16,0.034
4038,A Very Long Engagement,2004,Drama,1319.0,18,0.064
3304,Confessions of a Dangerous Mind,2002,Comedy,1252.0,36,0.085
3174,Panic Room,2002,Crime,5362.0,1,0.086
12265,Inside Out 2,2024,Animation,6626.0,1,0.086
542,Malcolm X,1992,Drama,1878.0,36,0.087
9848,Barbie,2023,Comedy,11028.0,5,0.094
12844,The Invitation,2016,Thriller,2831.0,7,0.099


Least robust communities (interpret with care):


,community_id,n_films,community_robustness,representative_films
25,25,185,0.378,My Brother and I (2023); The Perfect Family (2021); Twin Falls Idaho (1999); Prodigies (2024); The Parent ...
31,31,120,0.423,After the Ball (2015); One True Loves (2023); Lullaby (2022); Anna's War (2018); The Model (2016); The Uni...
37,37,79,0.431,Moebius (1996); Unstoppable (2010); Railroad Man (1999); Red Line (2013); Boiling Point (1993); Lucky Day ...
29,29,147,0.466,Old Man (2022); 1313: Bigfoot Island (2012); The High Frontier (2016); The Small Town (1997); The Monster ...
9,9,656,0.491,Ma che bella sorpresa (2015); Thesis on a Homicide (2013); The Substitute (2022); After (2019); 18 Years O...
32,32,114,0.503,Mandragora (1997); Oleg (2019); Transylvania (2006); Rafiki (2018); Easy Money III: Life Deluxe (2013); Da...
20,20,252,0.507,At Any Price (2012); Breakdown (1997); Born to Race (2011); Swerve (2012); Drive (2011); Race to Witch Mou...
28,28,149,0.513,Boys (2003); Boyhood (2014); Alone With Her Dreams (2019); Stockholm (2013); Liberty Heights (1999); Sparr...


## Outputs and checks

* `allow_corpus_change` stays `False` in Notebook 01, so the safety guard protects this network.
* `outputs/tables/community_label_carryover.csv`: communities marked *carried* keep their label; *carried – review* and *new* are checked in Notebook 04, which starts from `community_labels_draft_v2.csv`.
* `network_diagnostics/edge_diagnostics_summary.csv`: the share of near-duplicate and shared-name edges.
* `network_diagnostics/consensus_communities.csv`: consensus diagnostics; the reproducibility ARI should be close to 1.
* Communities with `community_robustness` well below 0.5 are described as tentative. Low-confidence films are candidates for the bridging analysis.